In [1]:
import pandas as pd
import numpy as np
import holidays
import pickle
from sklearn.preprocessing import OneHotEncoder
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from category_encoders import TargetEncoder
import nltk

In [2]:
kijkcijfers = pd.read_csv('./data2/processed/kijkcijfers_weerdata.csv')

# Basic features toevoegen

Functies om bepaalde feature engineering delen eenvoudig te kunnen toepassen

In [3]:
# Hulpfunctie om seizoen te bepalen
def get_season(date):
    if date.month in [3, 4, 5]:
        return 'spring'
    elif date.month in [6, 7, 8]:
        return 'summer'
    elif date.month in [9, 10, 11]:
        return 'autumn'
    else:
        return 'winter'

# Functie om features op basis van timestamp te maken
def timestamp_feature_engineering(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    df['season'] = df['timestamp'].apply(get_season)

    # weekday toevoegen
    df['weekday'] = df['timestamp'].dt.weekday

    # uur toevoegen
    df['hour'] = df['timestamp'].dt.hour

    # dag toevoegen
    df['day'] = df['timestamp'].dt.day

    # maand toevoegen
    df['month'] = df['timestamp'].dt.month

    # isWeekend toevoegen
    df['isWeekend'] = df['weekday'].apply(lambda x: 1 if x in [4, 5] else 0)

    # isPrimeTime toevoegen
    df['isPrimeTime'] = df['hour'].apply(lambda x: 1 if x >= 18 and x <= 21 else 0)

    # Sunrise en sunset hour toevoegen ipv datetime
    df['sunrise_hour'] = df['sunrise'].apply(lambda x: pd.to_datetime(x).hour)
    df['sunset_hour'] = df['sunset'].apply(lambda x: pd.to_datetime(x).hour)

    df.drop(['sunrise', 'sunset'], axis=1, inplace=True)

# Functie om lag features toe te voegen
def add_lag(df, n):
    for i in range(1, n+1):
        df[f'viewers_lag{i}'] = df.sort_values('timestamp').groupby('program')['viewers'].shift(i).ffill()
    return df

# Functie om one hot encoding toe te voegen
def category_encoding(df):
    # Categorical features
    cats = ['weather_code', 'season', 'channel', 'day_of_week']

    cat_encoder = OneHotEncoder(handle_unknown='ignore')
    df_cat = df[cats]
    one_hot = cat_encoder.fit_transform(df_cat).toarray()

    one_hot_df = pd.DataFrame(one_hot, columns=cat_encoder.get_feature_names_out(), index=df_cat.index)
    
    df = df.drop(cats, axis=1)
    df = pd.concat([df, one_hot_df], axis=1)

    with open('./models2/one_hot_encoder.pkl', 'wb') as f:
        pickle.dump(cat_encoder, f)
    
    return df

# Functie om stopwoorden uit program naam te verwijderen
def remove_program_stopwords(df):
    nltk.download('stopwords')

    stop_words_nl = set(stopwords.words('dutch'))
    stop_words_en = set(stopwords.words('english'))
    stop_words = stop_words_nl.union(stop_words_en)

    extra_stopwoorden = {'aflevering', 'herhaling', 'van', 'het', 'de', 'with', '?', ','}
    stop_words.update(extra_stopwoorden)

    def remove_stopwords(text):
        words = text.lower().split()
        words_filtered = [word for word in words if word not in stop_words]
        return ' '.join(words_filtered)

    df['program'] = df['program'].apply(remove_stopwords)
    return df

# Functie om tf-idf vectorisatie toe te voegen
def tfidf_vectorization(df):
    vectorizer = TfidfVectorizer(max_features=100)
    tfidf_matrix = vectorizer.fit_transform(df['program'])
    
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out(), index=df.index)
    
    df = pd.concat([df, tfidf_df], axis=1)

    with open('./models2/tfidf_vectorizer.pkl', 'wb') as f:
        pickle.dump(vectorizer, f)

    return df

# Functie om target encoding toe te voegen
def target_encoding(df, column, target):
    target_encoder = TargetEncoder(cols=[column])
    target_encoder.fit(df[column], df[target])
    df[f'{column}_target_enc'] = target_encoder.transform(df[column], df[target])
    
    # df.drop(column, axis=1, inplace=True)

    with open('./models2/target_encoder.pkl', 'wb') as f:
        pickle.dump(target_encoder, f)
    
    return df

In [4]:
# Timestamp feature engineering toepassen
timestamp_feature_engineering(kijkcijfers)

# Lag features toevoegen, en daarna lege waarden verwijderen
add_lag(kijkcijfers, 2)

kijkcijfers.dropna(inplace=True)

# One hot encoding toepassen
kijkcijfers = category_encoding(kijkcijfers)

# Stopwoorden verwijderen uit program's
remove_program_stopwords(kijkcijfers)

# TF-IDF vectorization toepassen
kijkcijfers = tfidf_vectorization(kijkcijfers)

# Target encoding toepassen
kijkcijfers = target_encoding(kijkcijfers, 'program', 'viewers')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dylan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
kijkcijfers_num = kijkcijfers.select_dtypes(include=[np.number])
corr_mx = kijkcijfers_num.corr()

print(corr_mx['viewers'].abs().sort_values(ascending=False).head(20))

viewers               1.000000
program_target_enc    0.827016
viewers_lag1          0.815045
viewers_lag2          0.781985
isPrimeTime           0.406806
thuis                 0.388283
channel_EEN           0.334165
uur                   0.241644
beroemd               0.222450
iedereen              0.215334
daylight_duration     0.173069
sunrise_hour          0.171940
temperature_2m        0.170096
season_summer         0.165941
sunset_hour           0.165632
13u                   0.160820
journaal              0.158179
day_of_week_5         0.155534
channel_Canvas        0.153826
isWeekend             0.153805
Name: viewers, dtype: float64


In [6]:
kijkcijfers.to_csv('./data2/feat_eng/kijkcijfers_weerdata.csv', index=False)

print("Feature engineering completed and saved to ./data2/feat_eng/kijkcijfers_weerdata.csv")

Feature engineering completed and saved to ./data2/feat_eng/kijkcijfers_weerdata.csv


# Verder Sentence Embedding proberen toe te passen op program kolom 

In [7]:
from sentence_transformers import SentenceTransformer

# Sentence Tranformer = ST
ST_model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = ST_model.encode(kijkcijfers['program'].tolist(), convert_to_tensor=False, show_progress_bar=True)

embeddings

C:\Users\dylan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 1908/1908 [01:09<00:00, 27.61it/s]


array([[-0.04972122,  0.05220046, -0.02193069, ...,  0.0038947 ,
         0.1261286 ,  0.04500864],
       [-0.03611445,  0.07150924,  0.00673143, ...,  0.00591934,
        -0.00463776,  0.00846623],
       [-0.07852581,  0.07642351,  0.0230902 , ...,  0.03275228,
        -0.0201158 ,  0.03704982],
       ...,
       [ 0.01168623,  0.05890074, -0.02174655, ..., -0.05414472,
         0.00260136, -0.03692932],
       [-0.06526288,  0.06124354, -0.01023662, ..., -0.02036745,
        -0.03254829, -0.01051877],
       [-0.07640638,  0.13453007,  0.00344516, ..., -0.01168494,
         0.02005988, -0.03309684]], dtype=float32)

In [8]:
embedding_columns = [f"embedding_{i}" for i in range(embeddings.shape[1])]

df_embeddings = pd.DataFrame(embeddings, columns=embedding_columns)

kijkcijfers = pd.concat([kijkcijfers, df_embeddings], axis=1)

In [9]:
kijkcijfers.head()

,timestamp,year,month,day,hour,program,duration_sec,viewers,temperature_2m,precipitation,...,embedding_374,embedding_375,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383
40,2016-10-03 20:14:23,2016.0,10.0,3.0,20.0,thuis,1530.0,1268561.0,15.3,0.0,...,0.069106,0.021666,0.037620,0.030973,-0.088024,0.034180,0.130373,0.003895,0.126129,0.045009
41,2016-10-03 19:47:08,2016.0,10.0,3.0,19.0,iedereen beroemd,1253.0,1111170.0,16.7,0.0,...,0.105392,-0.011500,0.020462,-0.008780,0.006860,0.037604,0.104025,0.032752,-0.020116,0.037050
42,2016-10-03 19:00:05,2016.0,10.0,3.0,19.0,7 uur-journaal,2683.0,1077321.0,16.7,0.0,...,0.058045,-0.017365,-0.072391,0.035651,-0.050965,0.031543,0.058760,0.005919,-0.004638,0.008466
43,2016-10-03 20:41:32,2016.0,10.0,3.0,20.0,spoed 24/7,2793.0,1063322.0,15.3,0.0,...,0.011243,-0.008143,-0.007556,0.056387,-0.063876,0.008667,-0.015383,-0.003461,0.088497,0.041827
44,2016-10-03 18:32:01,2016.0,10.0,3.0,18.0,blokken,1611.0,754051.0,17.8,0.0,...,0.057834,-0.031971,-0.052170,-0.005934,0.001250,-0.064510,0.048261,-0.018517,0.036861,0.050687


In [10]:
kijkcijfers_num = kijkcijfers.select_dtypes(include=[np.number])
corr_mx = kijkcijfers_num.corr()

print(corr_mx['viewers'].abs().sort_values(ascending=False).head(20))

viewers               1.000000
program_target_enc    0.827016
viewers_lag1          0.815045
viewers_lag2          0.781985
isPrimeTime           0.406806
thuis                 0.388283
channel_EEN           0.334165
embedding_11          0.289080
embedding_310         0.273080
embedding_102         0.265025
embedding_174         0.256378
embedding_299         0.255415
embedding_303         0.249388
embedding_254         0.248760
uur                   0.241644
embedding_363         0.241628
embedding_231         0.237324
embedding_367         0.233349
embedding_155         0.227371
embedding_256         0.226239
Name: viewers, dtype: float64


In [11]:
kijkcijfers.to_csv('./data2/feat_eng/kijkcijfers_weerdata_met_sentence_embedding.csv', index=False)